# Note:
- The data from the [Chicago Data Portal](https://data.cityofchicago.org/browse?category=Public+Safety&sortBy=most_accessed&page=1&pageSize=20) and Crime Data set were enriched using multiple datasets from the portal. We initially stored them in the PostgreSQL database to generate the enriched dataset by joining multiple datasets using the_geom, but we discovered inconsistencies in the police beat, district, and sector fields. All missing fields were determined using multiple fields to generate the most accurate information, but there may be errors during the data wrangling process.

- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation using data from other sources to fill corresponding NaN entries in location-based fields.

In [1]:
# import libraries
from platform import python_version
import sys
import time
import pandas as pd
import pyarrow as pa
import pyarrow.ipc as ipc
import numpy as np
import re

# python source path
sys.path.append('../Src/')

# python
import utils
import geo
import geo_dict

# seed
SEED = 1776

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# capture time
start = time.time()

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0


## Read Data
- The Chicago Crime Data contains Crime, Arrest, IUCR, Neighborhood, and Police Beat datasets from the Chicago Crime Portal.

In [2]:
# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# reset options
# pd.reset_option('display.max_columns')

In [3]:
# load using pyarrow for performance (crime data is joined between crime & neighborhood & police using geom)
df_crime = pd.read_csv("../Data/chicago_crimes_export.csv", engine="pyarrow", dtype_backend="pyarrow")
# copy
df_orig = df_crime.copy

In [4]:
df_crime.head()

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location,x_coordinate,y_coordinate
0,JJ533661,2025-12-24 00:00:00,009XX E 132ND PL,0560,ASSAULT,SIMPLE,N,ASSAULT,SIMPLE,RESIDENCE,f,t,533,5,10,54,2025,2025-12-31 15:41:59,08A,60827,48551330.6029,Riverdale,RIVERDALE,98389497.4143,5,3,533,54,RIVERDALE,98389497.4143,"(41.655276697,-87.597292828)",1185392,1817834
1,JJ533863,2025-12-24 00:00:00,043XX W IRVING PARK RD,1750,OFFENSE INVOLVING CHILDREN,CHILD ABUSE,N,OFFENSE INVOLVING CHILDREN,CHILD ABUSE,RESIDENCE,f,f,1722,17,45,16,2025,2025-12-31 15:41:59,08B,60641,113903341.241,Irving Park,"IRVING PARK,AVONDALE",89611382.3106,17,3,1731,16,IRVING PARK,89611382.3106,"(41.953552346,-87.7357763)",1146792,1926231
2,JJ537313,2025-12-24 00:00:00,072XX S RHODES AVE,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,I,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,f,f,323,3,6,69,2025,2025-12-31 15:41:59,07,60619,167872012.337,Grand Crossing,"SOUTH SHORE, GRAND CROSSING",98853167.7093,3,2,323,69,GREATER GRAND CROSSING,98853167.7093,"(41.763419509,-87.611581254)",1181157,1857207
3,JJ533996,2025-12-24 00:00:00,031XX S KEDVALE AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,N,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,t,t,1031,10,22,30,2025,2025-12-31 15:41:59,08B,60623,155285530.844,Little Village,LITTLE VILLAGE,127998297.819,10,3,1031,30,SOUTH LAWNDALE,127998297.867,"(41.836255581,-87.727988288)",1149212,1883503
4,JJ539138,2025-12-24 00:00:00,045XX W HARRISON ST,0820,THEFT,$500 AND UNDER,I,THEFT,$500 AND UNDER,SIDEWALK,f,f,1131,11,28,26,2025,2025-12-31 15:41:59,06,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## Describe Data

In [5]:
df_crime.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
case_number,8470050,8469443,HZ140230,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,8470050,NaN,NaN,NaN,2011-08-03 09:16:56,2001-01-01 00:00:00,2005-06-13 19:30:00,2010-06-06 09:19:00,2017-05-19 15:50:45,2025-12-24 00:00:00,NaN
block,8470050,65510,001XX N STATE ST,17067,NaN,NaN,NaN,NaN,NaN,NaN,NaN
iucr,8470050,418,0820,679332,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_description,8455087,32,THEFT,1790788,NaN,NaN,NaN,NaN,NaN,NaN,NaN
secondary_description,8455087,369,SIMPLE,996364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
index_code,8455087,2,N,5004675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_type,8470050,34,THEFT,1798858,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,8470050,569,SIMPLE,996364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_description,8454712,218,STREET,2213499,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# duplicate record
df_crime[df_crime.case_number == 'HJ590004'].sort_values('updated_on')

,case_number,date,block,iucr,primary_description,secondary_description,index_code,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,year,updated_on,fbi_code,zip_code,zip_code_area,primary_neighborhood,secondary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location,x_coordinate,y_coordinate
7184838,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)",1172943,1878840
7184839,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)",1172943,1878840
7184840,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)",1172943,1878840
7184841,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)",1172943,1878840
7184842,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)",1172943,1878840
7184843,HJ590004,2003-08-27 08:35:00,039XX S WALLACE ST,0110,HOMICIDE,FIRST DEGREE MURDER,I,HOMICIDE,FIRST DEGREE MURDER,WAREHOUSE,t,f,925,9,11,61,2003,2022-09-19 15:41:05,01A,60609,213490324.899,New City,BACK OF THE YARDS,134636963.254,9,2,925,61,NEW CITY,134636963.254,"(41.822967958,-87.641048569)",1172943,1878840


## Data Wrangle
- Chicago's `IUCR` codes (Illinois Uniform Crime Reporting) are four-digit codes for classifying crimes, with the Chicago Police Department (CPD) using over 400, including FBI Index Offenses (homicide, robbery, theft) and Non-Index offenses (vandalism, weapons violations)
- Chicago has `50 wards`, each represented by an alderperson, with boundaries redrawn every eight years
- The Chicago Police Department (CPD) divides the city into `22 Districts`, which are further broken down into smaller patrol zones called `Beats`, with specific 4-digit numbers for each area
- Chicago is divided into `77 official Community Areas`

In [7]:
# Use a single Arrow string & int64 type instance to save memory
arrow_string = pd.ArrowDtype(pa.string())
arrow_int64 = pd.ArrowDtype(pa.int64())

In [8]:
# Convert & Force pyarrow
df_crime['community_name'] = df_crime['community_name'].str.title()
df_crime['community_name'] = df_crime['community_name'].astype(arrow_string)

In [9]:
# get dupes
dupes = df_crime.duplicated(keep='last')
# any duplicates
if dupes.any():
    print(f"Number of Duplicates: {dupes.sum():,}")
else:
    print("No Duplicates")

Number of Duplicates: 178


In [10]:
# num of rows before 
before = df_crime.shape[0]
# remove any duplicates
df_crime = df_crime.sort_values('updated_on').drop_duplicates(keep='last').reset_index(drop=True)
# num of rows after 
after = df_crime.shape[0]
print(f"Duplicate Rows Removed: {(before - after):,}")
print(f"Shape: {df_crime.shape[0]:,} Rows & {df_crime.shape[1]:,} Columns")

Duplicate Rows Removed: 178
Shape: 8,469,872 Rows & 33 Columns


In [11]:
# drop updated_on
df_crime = df_crime.drop(columns=['updated_on', 'case_number'])

In [12]:
# display number of unique values
for i in df_crime.columns:
    print(f"{i}: {df_crime[i].nunique():,}")

date: 3,545,853
block: 65,510
iucr: 418
primary_description: 32
secondary_description: 369
index_code: 2
primary_type: 34
description: 569
location_description: 218
arrest: 2
domestic: 2
beat: 305
district: 24
ward: 50
community_area: 78
year: 25
fbi_code: 26
zip_code: 59
zip_code_area: 59
primary_neighborhood: 98
secondary_neighborhood: 78
neighborhood_area: 98
p_district: 22
p_sector: 4
p_beat: 274
community_code: 77
community_name: 77
ca_community_area: 77
location: 910,551
x_coordinate: 79,349
y_coordinate: 130,434


#### New Feature(s)

In [13]:
# Add Month & Day of the Week
months = ['January', 'February', 'March', 'April', 'May', 'June', 
          'July', 'August', 'September', 'October', 'November', 'December']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Extract integers and map them
# .dt.month returns 1-12, so we subtract 1 for 0-based indexing
df_crime['month'] = np.array(months)[df_crime['date'].dt.month.values - 1]

# .dt.dayofweek returns 0-6 (0 is Monday)
df_crime['day_of_week'] = np.array(days)[df_crime['date'].dt.dayofweek.values]

# convert to pyarrow
df_crime['month'] = df_crime['month'].astype(arrow_string)
df_crime['day_of_week'] = df_crime['day_of_week'].astype(arrow_string)

In [14]:
# Map to strings first
df_crime['quarter'] = df_crime['date'].dt.quarter.map({1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}).astype(arrow_string)
# combine
df_crime['year_quarter'] = (df_crime['year'].astype("string[pyarrow]") + "-" + df_crime['quarter'])
# Force pyarrow datatype
df_crime['year_quarter'] = df_crime['year_quarter'].astype(arrow_string)

| Interval (Inclusive, Exclusive) | Mathematical Notation | Label        | Hours Included    |
|--------------------------------|----------------------|--------------|-------------------|
| 1st: 0 to 4                    | \([0, 4)\)          | Late Night   | 0, 1, 2, 3       |
| 2nd: 4 to 8                    | \([4, 8)\)          | Early Morning| 4, 5, 6, 7       |
| 3rd: 8 to 12                   | \([8, 12)\)         | Morning      | 8, 9, 10, 11     |
| 4th: 12 to 16                  | \([12, 16)\)        | Afternoon    | 12, 13, 14, 15   |
| 5th: 16 to 20                  | \([16, 20)\)        | Evening      | 16, 17, 18, 19   |
| 6th: 20 to 24                  | \([20, 24)\)        | Night        | 20, 21, 22, 23   |

In [15]:
# Get the hours as a PyArrow-backed integer
hours = df_crime['date'].dt.hour.values

# Use np.digitize for ultra-fast binning (vectorized)
# bins: [0, 4, 8, 12, 16, 20, 24]
# digitize returns 1 for 0-3, 2 for 4-7, etc.
bin_indices = np.digitize(hours, bins=[4, 8, 12, 16, 20])

# Map indices to labels
time_labels = np.array(['Late Night', 'Early Morning', 'Morning', 'Afternoon', 'Evening', 'Night'])
df_crime['time_of_day'] = time_labels[bin_indices]

# Final cast to string[pyarrow]
df_crime['time_of_day'] = df_crime['time_of_day'].astype(arrow_string)

#### FBI Code Mapping

In [16]:
# display fbi_code
print(sorted(df_crime['fbi_code'].unique()))

['01A', '01B', '02', '03', '04A', '04B', '05', '06', '07', '08A', '08B', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '22', '24', '26']


In [17]:

# determine specific values in the data that are missing from the mapping dictionary
df_crime.loc[~df_crime["fbi_code"].isin(geo_dict.fbi_codes.keys()), "fbi_code" ].unique()

<ArrowExtensionArray>
[]
Length: 0, dtype: string[pyarrow]

In [18]:
df_crime[["fbi_code_desc", "fbi_index_code"]] = pd.DataFrame({
    "fbi_code_desc": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["desc"]),
    "fbi_index_code": df_crime["fbi_code"].map(lambda x: geo_dict.fbi_codes[x]["is_index"])
})

# convert to arrow datatype
df_crime["fbi_code_desc"] = df_crime["fbi_code_desc"].astype(arrow_string)
df_crime["fbi_index_code"] = df_crime["fbi_index_code"]

# drop fbi_code & relared descriptions
df_crime = df_crime.drop(columns=['iucr', 'primary_description', 'secondary_description', 'primary_type', 'fbi_code', 'index_code'])

In [19]:
# df_crime[['iucr','primary_description','secondary_description','description','fbi_code']][df_crime['fbi_code'] == '01A'].sample(5)
df_crime[['fbi_code_desc','description', 'location_description', 'domestic', 'fbi_index_code']].sample(n=5, random_state=SEED)

,fbi_code_desc,description,location_description,domestic,fbi_index_code
2248216,Miscellaneous Non-Index Offenses,HARASSMENT BY TELEPHONE,RESIDENCE,t,False
2467513,Fraud,THEFT OF LABOR/SERVICES,CTA PLATFORM,f,False
6123113,Larceny – Theft,ATTEMPT THEFT,STREET,f,True
6472970,Vandalism,TO PROPERTY,RESIDENCE,t,False
5868892,Larceny – Theft,OVER $500,STREET,f,True


#### Neighborhood Compare

In [20]:
# 1. Create a boolean mask using the underlying Arrow arrays (fastest)
mask = df_crime['primary_neighborhood'].str.lower() != df_crime['secondary_neighborhood'].str.lower()

# 2. Slice, Drop Duplicates, and then Sort
# Reducing the rows BEFORE sorting is the key to speed.
diff_neighborhoods = (
    df_crime.loc[mask, ['ward','community_area','zip_code','beat','district','primary_neighborhood','secondary_neighborhood']]
    .drop_duplicates()
    .sort_values('primary_neighborhood')
)

# display
print(diff_neighborhoods.sample(n=10, random_state=SEED).sort_values('primary_neighborhood').to_string())

         ward  community_area  zip_code  beat  district primary_neighborhood       secondary_neighborhood
2084729    11              35     60616   211         2        Armour Square      ARMOUR SQUARE,CHINATOWN
493630      8              45     60617   414         4          Avalon Park  AVALON PARK,CALUMET HEIGHTS
4436855    25              33     60616  2111         9            Chinatown      ARMOUR SQUARE,CHINATOWN
51372      23              57     60632   815         8       Garfield Ridge               MIDWAY AIRPORT
4093682     3              40     60615   232         9      Grand Boulevard                  BRONZEVILLE
43629    <NA>            <NA>     60620   623         6       Grand Crossing  SOUTH SHORE, GRAND CROSSING
523182     31              20     60641  2524        25              Hermosa      BELMONT CRAIGIN,HERMOSA
530606     31              16     60641  1731        17              Hermosa      BELMONT CRAIGIN,HERMOSA
2955457     5              43     60619   411 

In [21]:
# 1. Create a boolean mask using the underlying Arrow arrays (fastest)
mask = df_crime['primary_neighborhood'].str.lower() != df_crime['community_name'].str.lower()

# 2. Slice, Drop Duplicates, and then Sort
# Reducing the rows BEFORE sorting is the key to speed.
diff_neighborhoods = (
    df_crime.loc[mask, ['ward','community_area', 'community_code','zip_code','beat','district','primary_neighborhood','community_name']]
    .drop_duplicates()
    .sort_values('primary_neighborhood')
)

# display
print(diff_neighborhoods.sample(n=10, random_state=SEED).sort_values('primary_neighborhood').to_string())

         ward  community_area  community_code  zip_code  beat  district primary_neighborhood      community_name
6246131     6              67              67     60636   734         7            Englewood      West Englewood
1000       28              26              26     60624  1122        11        Garfield Park  West Garfield Park
6245860    27              27              27     60612  1135        11        Garfield Park  East Garfield Park
4771        1              24              24     60647  1421        14        Humboldt Park           West Town
2744202    28              29              24     60647  1023        14        Humboldt Park           West Town
43507    <NA>            <NA>              28     60608  1224        12    Little Italy, UIC      Near West Side
3944188    42               8               8     60654  1834         1          River North     Near North Side
7993024    44               6               8     60611  1924        18        Streeterville    

In [22]:
# drop secondary_neighborhood
df_crime = df_crime.drop(columns=['secondary_neighborhood'])
# display
df_crime.sample(n=5, random_state=SEED)

,date,block,description,location_description,arrest,domestic,beat,district,ward,community_area,year,zip_code,zip_code_area,primary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location,x_coordinate,y_coordinate,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
2248216,2011-12-19 07:11:00,0000X E ROOSEVELT RD,HARASSMENT BY TELEPHONE,RESIDENCE,f,t,132,1,2,32,2011,60605,36301276.0113,Near South Side,34252582.7003,1,3,131,33,Near South Side,49769639.4541,"(41.867414944,-87.627231186)",1176576,1895067,December,Monday,Q4,2011-Q4,Early Morning,Miscellaneous Non-Index Offenses,False
2467513,2011-03-26 19:01:00,008XX N STATE ST,THEFT OF LABOR/SERVICES,CTA PLATFORM,f,f,1832,18,42,8,2011,60611,23506055.7112,Rush & Division,3489575.8016,18,3,1833,8,Near North Side,76675895.9728,"(41.896888586,-87.628203192)",1176223,1905805,March,Saturday,Q1,2011-Q1,Evening,Fraud,False
6123113,2018-05-04 06:14:00,074XX S HALSTED ST,ATTEMPT THEFT,STREET,t,f,733,7,17,68,2018,60621,104746821.155,Englewood,173600015.009,7,3,733,68,Englewood,85652323.0826,"(41.759402215,-87.644268517)",1172251,1855670,May,Friday,Q2,2018-Q2,Early Morning,Larceny – Theft,True
6472970,2019-08-25 05:00:00,100XX S EBERHART AVE,TO PROPERTY,RESIDENCE,f,t,511,5,9,49,2019,60628,345241691.573,Roseland,134313706.73,5,1,511,49,Roseland,134313706.73,"(41.712270352,-87.61146519)",1181345,1838569,August,Sunday,Q3,2019-Q3,Early Morning,Vandalism,False
5868892,2002-07-08 11:15:00,017XX N CALIFORNIA AVE,OVER $500,STREET,f,f,1421,14,1,24,2002,60647,106052287.436,Humboldt Park,125010425.593,14,2,1421,24,West Town,127562904.597,"(41.91293292,-87.697064224)",1157432,1911505,July,Monday,Q3,2002-Q3,Morning,Larceny – Theft,True


#### Datatype Change (Boolean)

In [23]:
# check for unique values
df_crime[['arrest','domestic']].apply(lambda s: s.unique())

,arrest,domestic
0,f,f
1,t,t


In [24]:
# convert to pyarrow boolean
df_crime[['arrest','domestic']] = (
    df_crime[['arrest','domestic']] # domestic: Domestic violence
        .apply(lambda col: col.map({'t': True, 'f': False}))
        .astype(bool)
)

In [25]:
# display null columns
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
community_area        613682    7.2455%
p_district            116810    1.3791%
p_sector              116810    1.3791%
p_beat                116810    1.3791%
primary_neighborhood  116237    1.3724%
neighborhood_area     116237    1.3724%
community_code        116237    1.3724%
community_name        116237    1.3724%
ca_community_area     116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
location               94303    1.1134%
x_coordinate           94303    1.1134%
y_coordinate           94303    1.1134%
location_description   15338    0.1811%
district                  47    0.0006%


#### Feature Information:
* A Chicago `ward` is one of 50 legislative districts, each represented by an elected Alderman on the City Council, serving as local government branches to provide city services, manage development, and reflect community demographics, with boundaries redrawn every 10 years based on census data.
* The `district` feature refers to the city's 22 police districts, which are geographic areas used to organize crime data.
* The `beat` feature in Chicago crime data identifies the smallest geographic police area (a beat) where a crime occurred.
* The `sector` refers to a specific geographic division used by the Chicago Police Department (CPD), where several smaller `beats` (police patrol areas) are grouped together to form a sector, which then rolls up into a larger `district`, providing a layered geographic context for analyzing crime trends.
* The `Community Area` feature refers to one of 77 distinct, officially defined, and geographically stable neighborhoods used for urban planning and statistical analysis. This feature allows categorizing crime incidents by location, enabling trend analysis and identifying high-crime areas.

In [26]:
# Nans
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
community_area        613682    7.2455%
p_district            116810    1.3791%
p_sector              116810    1.3791%
p_beat                116810    1.3791%
primary_neighborhood  116237    1.3724%
neighborhood_area     116237    1.3724%
community_code        116237    1.3724%
community_name        116237    1.3724%
ca_community_area     116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
location               94303    1.1134%
x_coordinate           94303    1.1134%
y_coordinate           94303    1.1134%
location_description   15338    0.1811%
district                  47    0.0006%


In [27]:
# display ward
utils.wrap_unique(df_crime, 'ward')

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
43, 44, 45, 46, 47, 48, 49, 50]
::::: Unique Count: 50 (+ 614,815 nulls)


In [28]:
# display district
utils.wrap_unique(df_crime, 'district')
utils.wrap_unique(df_crime, 'p_district')

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24,
25, 31]
::::: Unique Count: 24 (+ 47 nulls)
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 22, 24, 25]
::::: Unique Count: 22 (+ 116,810 nulls)


In [29]:
# https://www.chicagopolice.org/statistics-data/crime-statistics/
# sector is represented as Area
print(sorted(df_crime.p_sector.fillna(-1).unique()))

[-1, 1, 2, 3, 5]


In [30]:
# display primary_neighborhood
utils.wrap_unique(df_crime, 'primary_neighborhood')

[Albany Park, Andersonville, Archer Heights, Armour Square, Ashburn, Auburn
Gresham, Austin, Avalon Park, Avondale, Belmont Cragin, Beverly, Boystown,
Bridgeport, Brighton Park, Bucktown, Burnside, Calumet Heights, Chatham, Chicago
Lawn, Chinatown, Clearing, Douglas, Dunning, East Side, East Village, Edgewater,
Edison Park, Englewood, Fuller Park, Gage Park, Galewood, Garfield Park,
Garfield Ridge, Gold Coast, Grand Boulevard, Grand Crossing, Grant Park,
Greektown, Hegewisch, Hermosa, Humboldt Park, Hyde Park, Irving Park, Jackson
Park, Jefferson Park, Kenwood, Lake View, Lincoln Park, Lincoln Square, Little
Italy, UIC, Little Village, Logan Square, Loop, Lower West Side, Magnificent
Mile, Mckinley Park, Millenium Park, Montclare, Morgan Park, Mount Greenwood,
Museum Campus, Near South Side, New City, North Center, North Lawndale, North
Park, Norwood Park, O'Hare, Oakland, Old Town, Portage Park, Printers Row,
Pullman, River North, Riverdale, Rogers Park, Roseland, Rush & Division,
Sau

In [31]:
# display community_name
utils.wrap_unique(df_crime, 'community_name')

[Albany Park, Archer Heights, Armour Square, Ashburn, Auburn Gresham, Austin,
Avalon Park, Avondale, Belmont Cragin, Beverly, Bridgeport, Brighton Park,
Burnside, Calumet Heights, Chatham, Chicago Lawn, Clearing, Douglas, Dunning,
East Garfield Park, East Side, Edgewater, Edison Park, Englewood, Forest Glen,
Fuller Park, Gage Park, Garfield Ridge, Grand Boulevard, Greater Grand Crossing,
Hegewisch, Hermosa, Humboldt Park, Hyde Park, Irving Park, Jefferson Park,
Kenwood, Lake View, Lincoln Park, Lincoln Square, Logan Square, Loop, Lower West
Side, Mckinley Park, Montclare, Morgan Park, Mount Greenwood, Near North Side,
Near South Side, Near West Side, New City, North Center, North Lawndale, North
Park, Norwood Park, Oakland, Ohare, Portage Park, Pullman, Riverdale, Rogers
Park, Roseland, South Chicago, South Deering, South Lawndale, South Shore,
Uptown, Washington Heights, Washington Park, West Elsdon, West Englewood, West
Garfield Park, West Lawn, West Pullman, West Ridge, West Town, W

In [32]:
# Determine if one is missing, the other is not
mask = df_crime.primary_neighborhood.isna() ^ df_crime.community_name.isna()
df_crime.loc[mask, ['primary_neighborhood', 'community_name']].head()

,primary_neighborhood,community_name


In [33]:
# display community_area
utils.wrap_unique(df_crime, 'community_area')

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41,
42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61,
62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]
::::: Unique Count: 78 (+ 613,682 nulls)


In [34]:
# display community_code
utils.wrap_unique(df_crime, 'community_code')

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22,
23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62,
63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]
::::: Unique Count: 77 (+ 116,237 nulls)


In [35]:
mask = df_crime.community_area.eq(0)
df_crime.loc[mask,].shape[0]

76

In [36]:
# Remove NaNs & remove duplicates & sort by zip_code
df_crime.loc[mask, ['community_area', 'community_code', 'zip_code','primary_neighborhood', 'community_name']]\
    .dropna().sort_values('zip_code').drop_duplicates()

,community_area,community_code,zip_code,primary_neighborhood,community_name
2985218,0,32,60602,Loop,Loop
2830261,0,32,60603,Loop,Loop
725357,0,56,60638,Garfield Ridge,Garfield Ridge
4523304,0,75,60655,Morgan Park,Morgan Park
2302361,0,76,60656,O'Hare,Ohare
2989071,0,10,60656,Norwood Park,Norwood Park
3622288,0,76,60666,O'Hare,Ohare


In [37]:
# build lookup: community_name → community_code (only valid rows)
lookup = (
    df_crime.loc[df_crime.community_area.ne(0), ['community_name', 'community_code']]
        .dropna().drop_duplicates()
        .set_index('community_name')['community_code'] # Set the index on community_name, display community_area column.
)

# fill zero community_area using the lookup & NaNs not in the lookup table
df_crime.loc[mask, 'community_area'] = (df_crime.loc[mask, 'community_name'].map(lookup))

In [38]:
# check
# all the '0' are updated
print(f"::::::: Community_area count: {df_crime.community_area.eq('0').sum()}\n")

df_crime.loc[mask, ['community_area', 'community_code', 'zip_code','primary_neighborhood', 'community_name']] \
    .dropna().sort_values('zip_code').drop_duplicates()



::::::: Community_area count: 0



,community_area,community_code,zip_code,primary_neighborhood,community_name
2985218,32,32,60602,Loop,Loop
2830261,32,32,60603,Loop,Loop
725357,56,56,60638,Garfield Ridge,Garfield Ridge
4523304,75,75,60655,Morgan Park,Morgan Park
2302361,76,76,60656,O'Hare,Ohare
2989071,10,10,60656,Norwood Park,Norwood Park
3622288,76,76,60666,O'Hare,Ohare


**Note:**
* Chicago’s crime data is recorded across a complex framework of overlapping jurisdictions, ranging from political districts to social neighborhoods. At the administrative level, the Chicago Police Department operates through a hierarchy of Districts and Beats. A Beat is the smallest geographic unit, assigned to a specific patrol car for community policing, while multiple Beats are grouped into a District managed by a central precinct. For example, Beats 2511, 2514, and 2521 all fall under the jurisdiction of District 025. Because these boundaries are drawn based on population density and response times rather than cultural history, they rarely align perfectly with the city’s social fabric.
* To provide a more stable lens for analysis, researchers utilize the city’s 77 Community Areas. Established in the 1920s by the University of Chicago, these fixed boundaries remain unchanged by political redistricting or postal updates, allowing for consistent longitudinal tracking of crime trends over decades. In contrast, Chicago’s 50 Wards are political entities redrawn every ten years to ensure equal population representation. Because Wards are subject to frequent shifts, they often bifurcate cohesive community areas and primary neighborhoods.
* Ultimately, "neighborhood" designations like "Albany Park" or "Irving Park" reflect social and historical identities rather than law-enforcement jurisdictions. Because these residential areas are often too large for a single patrol car to cover, a single neighborhood is frequently split across multiple Police Beats. This misalignment means that a single criminal incident may be categorized differently depending on whether the analyst is looking through a political (Ward), statistical (Community Area), or operational (Police District) lens.

**NaN Note:**
- We designate the primary Crime dataset as the authoritative source of truth. To ensure consistency and address missing values, we perform internal imputation by using the data from other sources to fill corresponding NaN entries within the location-based fields.

In [39]:
df_crime.head()

,date,block,description,location_description,arrest,domestic,beat,district,ward,community_area,year,zip_code,zip_code_area,primary_neighborhood,neighborhood_area,p_district,p_sector,p_beat,community_code,community_name,ca_community_area,location,x_coordinate,y_coordinate,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code
0,2001-12-30 02:30:00,073XX S YATES BLVD,AUTOMOBILE,STREET,False,False,334,3,7,43,2001,60649,80526075.8505,South Shore,81812716.3904,3,3,334,43,South Shore,81812716.3958,"(41.76145747,-87.566281678)",1193522,1856599,December,Sunday,Q4,2001-Q4,Late Night,Motor Vehicle Theft,True
1,2001-12-24 12:30:00,045XX N CLARENDON AVE,FRAUD OR CONFIDENCE GAME,APARTMENT,False,False,2313,19,46,3,2001,60640,77305245.6358,Uptown,65095642.836,19,1,1914,3,Uptown,65095642.7289,"(41.965220666,-87.650159176)",1170047,1930657,December,Monday,Q4,2001-Q4,Afternoon,Fraud,False
2,2001-12-23 22:30:00,028XX W 19TH ST,AUTOMOBILE,STREET,False,False,1022,10,12,30,2001,60623,155285530.844,Little Village,127998297.819,10,2,1022,30,South Lawndale,127998297.867,"(41.855402607,-87.698559781)",1157180,1890538,December,Sunday,Q4,2001-Q4,Night,Motor Vehicle Theft,True
3,2001-12-23 02:30:00,026XX W CHICAGO AVE,AUTOMOBILE,STREET,False,False,1311,12,26,24,2001,60622,70853834.1569,Ukrainian Village,10622385.5776,12,1,1211,24,West Town,127562904.597,"(41.895809634,-87.691772476)",1158919,1905276,December,Sunday,Q4,2001-Q4,Late Night,Motor Vehicle Theft,True
4,2001-12-27 19:00:00,031XX N MILWAUKEE AVE,"TRUCK, BUS, MOTOR HOME",OTHER,False,False,2523,25,31,21,2001,60618,141235867.463,Avondale,55290595.482,25,2,2523,21,Avondale,55290595.473,"(41.939022817,-87.723977703)",1150039,1920959,December,Thursday,Q4,2001-Q4,Evening,Motor Vehicle Theft,True


#### Convert to string
- Add padding if required

In [40]:
# change to string and must be three char length
cols = ['district', 'p_district', 'beat', 'p_beat', 'ward', 'community_area', 'p_sector', 'year', 'zip_code']

# iterate cols
for col in cols:

    if col in ['district', 'p_district']:
        # Fill NAs and convert to a standard string for the zfill operation
        # https://www.chicagopolice.org/statistics-data/crime-statistics/
        df_crime[col] = df_crime[col].astype("string").str.zfill(3)
    elif col in ['ward', 'community_area']:
        df_crime[col] = df_crime[col].astype("string").str.zfill(2)
    else:
        # Fill NAs and convert to a standard string
        df_crime[col] = df_crime[col].astype("string")
        
    # Force ArrowDtype
    df_crime[col] = df_crime[col].astype(arrow_string)

df_crime[cols].sample(5)

,district,p_district,beat,p_beat,ward,community_area,p_sector,year,zip_code
7735668,015,015,1522,1522,29,25,2,2024,60644
8362196,024,024,2432,2432,49,01,3,2025,60626
4415715,020,020,2024,2024,48,03,2,2006,60640
8339473,019,019,1923,1923,44,06,2,2025,60613
72948,020,020,2031,2031,<NA>,<NA>,3,2001,60625


### NaNs & Invalid Data

#### Community

In [41]:
print(df_crime.columns)

Index(['date', 'block', 'description', 'location_description', 'arrest',
       'domestic', 'beat', 'district', 'ward', 'community_area', 'year',
       'zip_code', 'zip_code_area', 'primary_neighborhood',
       'neighborhood_area', 'p_district', 'p_sector', 'p_beat',
       'community_code', 'community_name', 'ca_community_area', 'location',
       'x_coordinate', 'y_coordinate', 'month', 'day_of_week', 'quarter',
       'year_quarter', 'time_of_day', 'fbi_code_desc', 'fbi_index_code'],
      dtype='object')


In [42]:
df_crime.community_area.isna().sum(), df_crime.community_code.isnull().sum(), df_crime.community_name.isnull().sum()

(np.int64(613692), np.int64(116237), np.int64(116237))

In [43]:
# Cross-column validation to handle missing community area data (Crime)
mask = (df_crime.community_area.isna() & df_crime.community_code.notna()
       & (df_crime.primary_neighborhood == df_crime.community_name)
       )
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sample(10)

Mis-match count: 474,776



,community_area,primary_neighborhood,community_code,community_name,zip_code
429686,<NA>,Logan Square,22,Logan Square,60647
357537,<NA>,Auburn Gresham,71,Auburn Gresham,60620
109273,<NA>,Archer Heights,57,Archer Heights,60632
87325,<NA>,Roseland,49,Roseland,60628
110308,<NA>,Riverdale,54,Riverdale,60827
315961,<NA>,Englewood,68,Englewood,60621
149045,<NA>,North Lawndale,29,North Lawndale,60624
5990880,<NA>,Chicago Lawn,66,Chicago Lawn,60629
351179,<NA>,Englewood,68,Englewood,60621
284662,<NA>,Englewood,68,Englewood,60621


In [44]:
# Update community_area NaNs
df_crime.loc[mask, 'community_area'] = df_crime.loc[mask, 'community_code']

In [45]:
# Check Update
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sample(5)

,community_area,primary_neighborhood,community_code,community_name,zip_code
431332,43,South Shore,43,South Shore,60649
5956567,29,North Lawndale,29,North Lawndale,60623
468347,46,South Chicago,46,South Chicago,60617
44811,68,Englewood,68,Englewood,60621
171303,2,West Ridge,2,West Ridge,60645


In [46]:
# Lets tackle the rest of the community area
# Determine if one is missing, the other is not
mask = df_crime.community_area.isna() ^ df_crime.community_code.isna()
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sample(10)

Mis-match count: 235,659



,community_area,primary_neighborhood,community_code,community_name,zip_code
856227,32,<NA>,<NA>,<NA>,<NA>
7739908,70,<NA>,<NA>,<NA>,<NA>
29257,25,<NA>,<NA>,<NA>,<NA>
6933375,73,<NA>,<NA>,<NA>,<NA>
407451,<NA>,Wicker Park,24,West Town,60622
6667478,33,<NA>,<NA>,<NA>,<NA>
69890,<NA>,United Center,28,Near West Side,60612
6797121,32,<NA>,<NA>,<NA>,<NA>
5977772,<NA>,River North,8,Near North Side,60642
309155,<NA>,River North,8,Near North Side,60642


In [47]:
#Cross-column validation to handle missing community area data (Crime)
mask = (df_crime.community_area.isna() & df_crime.community_code.notnull()
       & (df_crime.primary_neighborhood != df_crime.community_name)
       )
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sort_values('zip_code').drop_duplicates().head(10)

Mis-match count: 129,169



,community_area,primary_neighborhood,community_code,community_name,zip_code
60853,<NA>,Millenium Park,32,Loop,60602
59356,<NA>,Millenium Park,32,Loop,60603
125161,<NA>,Grant Park,32,Loop,60603
45673,<NA>,Grant Park,32,Loop,60604
44298,<NA>,Museum Campus,33,Near South Side,60605
44377,<NA>,Printers Row,32,Loop,60605
44992,<NA>,Grant Park,32,Loop,60605
45158,<NA>,West Loop,28,Near West Side,60606
61239,<NA>,River North,8,Near North Side,60606
43759,<NA>,West Loop,28,Near West Side,60607


In [48]:
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']] \
    .sort_values('zip_code').drop_duplicates().tail(10)

,community_area,primary_neighborhood,community_code,community_name,zip_code
43535,<NA>,River North,8,Near North Side,60654
49024,<NA>,Rush & Division,8,Near North Side,60654
43736,<NA>,O'Hare,76,Ohare,60656
43990,<NA>,Boystown,6,Lake View,60657
48039,<NA>,Wrigleyville,6,Lake View,60657
51529,<NA>,"Sauganash,Forest Glen",12,Forest Glen,60659
44191,<NA>,West Loop,28,Near West Side,60661
44629,<NA>,Greektown,28,Near West Side,60661
2759565,<NA>,O'Hare,76,Ohare,60666
44740,<NA>,Galewood,25,Austin,60707


In [49]:
# Update community_area with community_code
df_crime.loc[mask, 'community_area'] = df_crime.loc[mask, 'community_code']
# Update community_name Ohare to O'hare
df_crime['community_name'] = df_crime['community_name'].replace({'Ohare':"O'Hare"})

In [50]:
df_crime.loc[mask, ['community_area','primary_neighborhood', 'community_code', 'community_name', 'zip_code']].sort_values('zip_code').drop_duplicates().tail(10)

,community_area,primary_neighborhood,community_code,community_name,zip_code
43535,8,River North,8,Near North Side,60654
49024,8,Rush & Division,8,Near North Side,60654
43736,76,O'Hare,76,O'Hare,60656
43990,6,Boystown,6,Lake View,60657
48039,6,Wrigleyville,6,Lake View,60657
51529,12,"Sauganash,Forest Glen",12,Forest Glen,60659
44191,28,West Loop,28,Near West Side,60661
44629,28,Greektown,28,Near West Side,60661
2759565,76,O'Hare,76,O'Hare,60666
44740,25,Galewood,25,Austin,60707


In [51]:
# Columns List
cols = ['community_area','primary_neighborhood', 'community_code', 'community_name', 
        'district', 'beat', 'ward', 'zip_code']
# NaNs
mask = df_crime.community_area.isna()
df_crime.loc[mask, cols].sample(10)

,community_area,primary_neighborhood,community_code,community_name,district,beat,ward,zip_code
824014,<NA>,<NA>,<NA>,<NA>,022,2212,<NA>,<NA>
42216,<NA>,<NA>,<NA>,<NA>,010,1022,<NA>,<NA>
8216339,<NA>,<NA>,<NA>,<NA>,005,522,<NA>,<NA>
824386,<NA>,<NA>,<NA>,<NA>,005,522,<NA>,<NA>
824699,<NA>,<NA>,<NA>,<NA>,004,432,<NA>,<NA>
41457,<NA>,<NA>,<NA>,<NA>,014,1424,<NA>,<NA>
826057,<NA>,<NA>,<NA>,<NA>,005,511,<NA>,<NA>
824331,<NA>,<NA>,<NA>,<NA>,022,2232,<NA>,<NA>
135028,<NA>,<NA>,<NA>,<NA>,003,314,<NA>,<NA>
41704,<NA>,<NA>,<NA>,<NA>,018,1834,<NA>,<NA>


In [52]:
mask = (df_crime.primary_neighborhood.isna() & df_crime.community_name.notna())
print(f"NaN count: {(mask.sum()):,}\n")

NaN count: 0



#### Police District

In [53]:
# columns to display
cols = ['ward', 'community_area', 'community_code', 'beat', 'p_beat', 'district', 'p_district',
        'p_sector', 'zip_code', 'primary_neighborhood', 'community_name']
# Invalid Police Districts
print(f"District 021 count:  {(df_crime.district.eq('021').sum()):,}")
print(f"District 031 count:  {(df_crime.district.eq('031').sum()):,}")

District 021 count:  4
District 031 count:  277


##### District 021

In [54]:
# For district 021
if (df_crime.district == '021').sum() <= 10:
    out = df_crime.loc[df_crime["district"].eq('021'), cols]
else:
    out = df_crime.loc[df_crime["district"].eq('021'), cols].sample(n=10, random_state=SEED)
# display
out

,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
5166842,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas
5198264,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas
5329390,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas
5521394,03,35,35,2112,211,021,002,1,60616,Douglas,Douglas


In [55]:
# Update invalid district 021
mask = df_crime["district"].eq('021')
df_crime.loc[mask, "district"] = df_crime.loc[mask, "p_district"]

##### District 031

In [56]:
# For district 031
if (df_crime.district == '031').sum() <= 10:
    out = df_crime.loc[df_crime["district"].eq('031'), cols]
else:
    out = df_crime.loc[df_crime["district"].eq('031'), cols].sample(n=10, random_state=SEED)
# display
out

,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
7161372,41,76,<NA>,1654,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6819956,41,76,<NA>,1653,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
2980778,19,75,75,2212,2212,031,022,1,60655,Morgan Park,Morgan Park
7427352,41,76,<NA>,1654,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6251856,<NA>,<NA>,<NA>,533,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6873034,41,76,<NA>,1654,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
6248146,<NA>,<NA>,<NA>,1651,<NA>,031,<NA>,<NA>,<NA>,<NA>,<NA>
2939376,07,46,46,422,422,031,004,2,60617,South Chicago,South Chicago
3941446,19,75,75,2212,2212,031,022,1,60655,Morgan Park,Morgan Park
4904830,19,75,75,2212,2212,031,022,1,60655,Morgan Park,Morgan Park


In [57]:
# Update invalid district 031
mask = df_crime["district"].eq('031')
df_crime.loc[mask, "district"] = df_crime.loc[mask, "p_district"]
# Update rest of invalid district 031 to NaNs
df_crime.loc[df_crime["district"] == '031', "district"] = pd.NA

In [58]:
# display district
utils.wrap_unique(df_crime, 'district')

[001, 002, 003, 004, 005, 006, 007, 008, 009, 010, 011, 012, 014, 015, 016, 017,
018, 019, 020, 022, 024, 025]
::::: Unique Count: 22 (+ 186 nulls)


##### Police District NaNs

In [59]:
# Cross columns check
mask = (df_crime.district.isna()) & df_crime.p_district.notnull()
print(f"Mis-match count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

Mis-match count: 47



,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
3827368,17,67,67,734,734,<NA>,007,3,60636,Englewood,West Englewood
3832149,01,24,24,1433,1433,<NA>,014,3,60622,Wicker Park,West Town
3825711,28,25,25,1113,1113,<NA>,011,1,60644,Austin,Austin
2872819,17,71,71,621,621,<NA>,006,2,60620,Auburn Gresham,Auburn Gresham
3864777,39,14,14,1722,1722,<NA>,017,2,60630,Albany Park,Albany Park


In [60]:
# Update District
df_crime.loc[mask, 'district'] = df_crime.loc[mask, 'p_district'] 
# Validate
df_crime.loc[mask, cols].sample(n=5, random_state=SEED)

,ward,community_area,community_code,beat,p_beat,district,p_district,p_sector,zip_code,primary_neighborhood,community_name
3827368,17,67,67,734,734,007,007,3,60636,Englewood,West Englewood
3832149,01,24,24,1433,1433,014,014,3,60622,Wicker Park,West Town
3825711,28,25,25,1113,1113,011,011,1,60644,Austin,Austin
2872819,17,71,71,621,621,006,006,2,60620,Auburn Gresham,Auburn Gresham
3864777,39,14,14,1722,1722,017,017,2,60630,Albany Park,Albany Park


In [61]:
# list of columns to display
cols = ['district', 'primary_neighborhood', 'community_code', 'community_name', 'p_sector', 'beat', 'zip_code']
mask = df_crime.district.isna()
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

NaN count: 139



,district,primary_neighborhood,community_code,community_name,p_sector,beat,zip_code
6780035,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>
7192131,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
7399443,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>
2799016,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
1848048,<NA>,<NA>,<NA>,<NA>,<NA>,1621,<NA>
7427352,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
6741458,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>
2767429,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
2382692,<NA>,<NA>,<NA>,<NA>,<NA>,1654,<NA>
6861735,<NA>,<NA>,<NA>,<NA>,<NA>,1653,<NA>


In [62]:
# Finding Unique Key for Update
key_cols = ['primary_neighborhood',	'community_code', 'community_name',	'p_sector', 'beat', 'zip_code']
target_col = 'district'
# fill district using Composite key
df_crime = geo.fill_from_composite_key(df_crime, key_cols, target_col).copy()

--- Fill Summary: Column (district) 0 missing values filled ---


In [63]:
# Create new district_location column
df_crime['district_location'] = df_crime['district'].map(geo_dict.cpd_districts).astype(arrow_string)

# Validate unmapped districts
df_crime.loc[df_crime["district_location"].isna(), "district"].unique()

<ArrowExtensionArray>
[<NA>]
Length: 1, dtype: string[pyarrow]

#### Sector Mapping

In [64]:
# Invert the mapping
cpd_sector = {
    dist: sector
    for sector, dists in geo_dict.cpd_sector.items() # Outer loop: looping over sector → list_of_districts
    for dist in dists # Inner loop: looping over each district inside that list
}

# Create the new sector column
df_crime["map_sector"] = df_crime["district"].map(cpd_sector).astype(arrow_string)

# Validate unmapped districts
df_crime.loc[df_crime["map_sector"].isna(), "district"].unique()

<ArrowExtensionArray>
[<NA>]
Length: 1, dtype: string[pyarrow]

In [65]:
# columns to display
cols = ['p_beat', 'district', 'p_district', 'p_sector', 'map_sector',
        'zip_code', 'primary_neighborhood', 'community_name']
# Cross-reference check - looking for disagreement
mask = (df_crime.p_sector.isna() ^ df_crime.map_sector.isna())
print(f"Disagree count: {mask.sum()}")
df_crime.loc[mask, cols].drop_duplicates().sample(n=10, random_state=SEED)

Disagree count: 116671


,p_beat,district,p_district,p_sector,map_sector,zip_code,primary_neighborhood,community_name
129609,<NA>,001,<NA>,<NA>,3,60602,Loop,Loop
6664,<NA>,005,<NA>,<NA>,2,<NA>,<NA>,<NA>
5867,<NA>,012,<NA>,<NA>,3,<NA>,<NA>,<NA>
9399,<NA>,004,<NA>,<NA>,2,<NA>,<NA>,<NA>
1710,<NA>,014,<NA>,<NA>,5,<NA>,<NA>,<NA>
1154587,<NA>,016,<NA>,<NA>,5,60631,<NA>,<NA>
7771292,<NA>,016,<NA>,<NA>,5,60646,"Sauganash,Forest Glen",Forest Glen
1713,<NA>,015,<NA>,<NA>,4,<NA>,<NA>,<NA>
9397,<NA>,011,<NA>,<NA>,4,<NA>,<NA>,<NA>
1203839,<NA>,022,<NA>,<NA>,2,60655,<NA>,<NA>


### DataFrame Maint

In [66]:
# Drop Columns
df_crime = df_crime.drop(columns=['p_beat', 'p_district', 'community_code', 'location', 'p_sector'])
# Rename Columns
df_crime = df_crime.rename(columns={'map_sector': 'sector', 'community_area': 'community_code',
                                    'ca_community_area' : 'community_area', 'primary_neighborhood': 'neighborhood'} )
# collapse fragmented blocks
df_crime = df_crime.copy()
# display
df_crime.head()

,date,block,description,location_description,arrest,domestic,beat,district,ward,community_code,year,zip_code,zip_code_area,neighborhood,neighborhood_area,community_name,community_area,x_coordinate,y_coordinate,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location,sector
0,2001-12-30 02:30:00,073XX S YATES BLVD,AUTOMOBILE,STREET,False,False,334,003,07,43,2001,60649,80526075.8505,South Shore,81812716.3904,South Shore,81812716.3958,1193522,1856599,December,Sunday,Q4,2001-Q4,Late Night,Motor Vehicle Theft,True,Grand Crossing,1
1,2001-12-24 12:30:00,045XX N CLARENDON AVE,FRAUD OR CONFIDENCE GAME,APARTMENT,False,False,2313,019,46,03,2001,60640,77305245.6358,Uptown,65095642.836,Uptown,65095642.7289,1170047,1930657,December,Monday,Q4,2001-Q4,Afternoon,Fraud,False,Town Hall,3
2,2001-12-23 22:30:00,028XX W 19TH ST,AUTOMOBILE,STREET,False,False,1022,010,12,30,2001,60623,155285530.844,Little Village,127998297.819,South Lawndale,127998297.867,1157180,1890538,December,Sunday,Q4,2001-Q4,Night,Motor Vehicle Theft,True,Ogden,4
3,2001-12-23 02:30:00,026XX W CHICAGO AVE,AUTOMOBILE,STREET,False,False,1311,012,26,24,2001,60622,70853834.1569,Ukrainian Village,10622385.5776,West Town,127562904.597,1158919,1905276,December,Sunday,Q4,2001-Q4,Late Night,Motor Vehicle Theft,True,Near West,3
4,2001-12-27 19:00:00,031XX N MILWAUKEE AVE,"TRUCK, BUS, MOTOR HOME",OTHER,False,False,2523,025,31,21,2001,60618,141235867.463,Avondale,55290595.482,Avondale,55290595.473,1150039,1920959,December,Thursday,Q4,2001-Q4,Evening,Motor Vehicle Theft,True,Grand Central,5


In [67]:
df_crime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469872 entries, 0 to 8469871
Data columns (total 28 columns):
 #   Column                Dtype                
---  ------                -----                
 0   date                  timestamp[s][pyarrow]
 1   block                 string[pyarrow]      
 2   description           string[pyarrow]      
 3   location_description  string[pyarrow]      
 4   arrest                bool                 
 5   domestic              bool                 
 6   beat                  string[pyarrow]      
 7   district              string[pyarrow]      
 8   ward                  string[pyarrow]      
 9   community_code        string[pyarrow]      
 10  year                  string[pyarrow]      
 11  zip_code              string[pyarrow]      
 12  zip_code_area         double[pyarrow]      
 13  neighborhood          string[pyarrow]      
 14  neighborhood_area     double[pyarrow]      
 15  community_name        string[pyarrow]      
 16  

In [68]:
# NaNs
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
neighborhood          116237    1.3724%
neighborhood_area     116237    1.3724%
community_name        116237    1.3724%
community_area        116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
x_coordinate           94303    1.1134%
y_coordinate           94303    1.1134%
location_description   15338    0.1811%
community_code          9747    0.1151%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


#### Neighborhood & Community

In [69]:
# list of columns to display
cols = ['neighborhood', 'neighborhood_area', 'community_code', 'community_name', 'community_area', 'zip_code', 'zip_code_area']
# Only zip codes not null
mask = df_crime.zip_code.notna() & (df_crime.neighborhood.isna() | df_crime.community_name.isna())
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].drop_duplicates().sort_values('zip_code')

NaN count: 71



,neighborhood,neighborhood_area,community_code,community_name,community_area,zip_code,zip_code_area
1154587,<NA>,<NA>,10,<NA>,<NA>,60631,107117592.044
1555052,<NA>,<NA>,02,<NA>,<NA>,60645,62181473.4847
1203839,<NA>,<NA>,72,<NA>,<NA>,60655,115380139.518
146079,<NA>,<NA>,<NA>,<NA>,<NA>,60656,89515884.2795
1034292,<NA>,<NA>,10,<NA>,<NA>,60656,89515884.2795
1197518,<NA>,<NA>,76,<NA>,<NA>,60656,89515884.2795


**Note:**
- Inconsistency in zip_code with related columns
- Only Update NaNs: neighborhood & community_name

In [70]:
# Determine Mapping
df_crime[['neighborhood', 'community_name', 'zip_code', 'zip_code_area']] \
    [(df_crime.zip_code.isin(['60631', '60645', '60655', '60656'  ]))] \
    .drop_duplicates().dropna().sort_values('zip_code')

,neighborhood,community_name,zip_code,zip_code_area
487,O'Hare,O'Hare,60631,107117592.044
497,Norwood Park,Norwood Park,60631,107117592.044
1077,Edison Park,Edison Park,60631,107117592.044
97,West Ridge,West Ridge,60645,62181473.4847
631,Rogers Park,Rogers Park,60645,62181473.4847
41,Mount Greenwood,Mount Greenwood,60655,115380139.518
641,Morgan Park,Morgan Park,60655,115380139.518
3765,Beverly,Beverly,60655,115380139.518
205,O'Hare,O'Hare,60656,89515884.2795
283,Norwood Park,Norwood Park,60656,89515884.2795


##### Neighborhood & Community_name

In [71]:
# mapping table
mapping = {'60631': 'Norwood Park', '60645': 'West Ridge', '60655': 'Mount Greenwood', '60656': "O'Hare"}

# Ensure Arrow string dtype for speed
zip_col = df_crime["zip_code"].astype(arrow_string)

# Build the mapped values
mapped_zip = zip_col.map(mapping)

# Mask: rows where neighborhood OR community_name is missing
nc_mask = df_crime["neighborhood"].isna() | df_crime["community_name"].isna()

# Apply mapping only to masked rows, keep original otherwise
df_crime.loc[nc_mask, "neighborhood"] = (
    mapped_zip.fillna(df_crime.loc[nc_mask, "neighborhood"])
)

df_crime.loc[nc_mask, "community_name"] = (
    mapped_zip.fillna(df_crime.loc[nc_mask, "community_name"])
)

In [72]:
# NaNs
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
neighborhood_area     116237    1.3724%
community_area        116237    1.3724%
zip_code              116181    1.3717%
zip_code_area         116181    1.3717%
neighborhood          116166    1.3715%
community_name        116166    1.3715%
x_coordinate           94303    1.1134%
y_coordinate           94303    1.1134%
location_description   15338    0.1811%
community_code          9747    0.1151%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


##### Community Code

In [73]:
# list of columns to display
cols = ['neighborhood', 'neighborhood_area', 'community_code', 'community_name', 'community_area', 'zip_code', 'zip_code_area']
# Only community codes null
mask = df_crime.community_code.notna() & (df_crime.neighborhood.isna() | df_crime.community_name.isnull())
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].drop_duplicates().sort_values('zip_code').sample(n=10, random_state=SEED)

NaN count: 106,422



,neighborhood,neighborhood_area,community_code,community_name,community_area,zip_code,zip_code_area
9380,<NA>,<NA>,19,<NA>,<NA>,<NA>,<NA>
2801,<NA>,<NA>,22,<NA>,<NA>,<NA>,<NA>
9473,<NA>,<NA>,65,<NA>,<NA>,<NA>,<NA>
9465,<NA>,<NA>,20,<NA>,<NA>,<NA>,<NA>
9422,<NA>,<NA>,57,<NA>,<NA>,<NA>,<NA>
4215,<NA>,<NA>,16,<NA>,<NA>,<NA>,<NA>
9488,<NA>,<NA>,42,<NA>,<NA>,<NA>,<NA>
6664,<NA>,<NA>,53,<NA>,<NA>,<NA>,<NA>
9446,<NA>,<NA>,43,<NA>,<NA>,<NA>,<NA>
9406,<NA>,<NA>,45,<NA>,<NA>,<NA>,<NA>


In [74]:
# Fill NaNs for neighborhood & community_name using community_code
df_crime = geo.fill_geo_from_lookup(df_crime, 'community_code', ['neighborhood', 'community_name']).copy()

--- Fill Summary (Key: community_code) ---
Column 'neighborhood': 106,422 rows filled.
Column 'community_name': 106,422 rows filled.


In [75]:
# Fill NaNs for zip_code using community_code
df_crime = geo.fill_geo_from_lookup(df_crime, 'community_code', ['zip_code']).copy()

--- Fill Summary (Key: community_code) ---
Column 'zip_code': 106,437 rows filled.


In [76]:
# Replaces : , -, and multi-spaces with a single space
def clean_locations(series):
    out = (
        series.str.replace(r'[\s:,,-]+', ' ', regex=True) # Combined delimiters to space
              .str.replace(r'\s*/\s*', '/', regex=True)  # Fix slashes
              .str.strip()
    )
    
    return out
# Apply regex
df_crime['location_description'] = clean_locations(df_crime['location_description'])

# Dictionary mapping (Vectorized replace)
mapping = {
    'NURSING HOME/RETIREMENT HOME': 'NURSING/RETIREMENT HOME', 
    'OTHER RAILROAD PROP/TRAIN DEPOT': 'OTHER RAILROAD PROPERTY/TRAIN DEPOT',
    'PARKING LOT/GARAGE(NON.RESID.)': 'PARKING LOT/GARAGE (NON RESIDENTIAL)',
    'POLICE FACILITY/VEH PARKING LOT': 'POLICE FACILITY/VEHICLE PARKING LOT',
    'POOLROOM': 'POOL ROOM', 
    'RESIDENCE YARD (FRONT/BACK)': 'RESIDENTIAL YARD (FRONT/BACK)',
    'TAXICAB': 'TAXI CAB',
    'VEHICLE OTHER RIDE SERVICE': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)',
    'VEHICLE OTHER RIDE SHARE SERVICE (E.G. UBER LYFT)': 'VEHICLE OTHER RIDE SHARE SERVICE (LYFT UBER ETC.)'
}

# Apply mapping first
df_crime['location_description'] = df_crime['location_description'].replace(mapping)

In [77]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
ward                  614815    7.2588%
neighborhood_area     116237    1.3724%
community_area        116237    1.3724%
zip_code_area         116181    1.3717%
x_coordinate           94303    1.1134%
y_coordinate           94303    1.1134%
location_description   15338    0.1811%
community_code          9747    0.1151%
zip_code                9744    0.1150%
neighborhood            9744    0.1150%
community_name          9744    0.1150%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


#### Ward

In [78]:
# list of columns to display
cols = ['ward', 'neighborhood', 'community_code', 'community_name', 'sector', 'district', 'beat', 'zip_code']
# Only Ward is NaN
mask = df_crime.ward.isna()
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

NaN count: 614,815



,ward,neighborhood,community_code,community_name,sector,district,beat,zip_code
6004402,<NA>,Austin,25,Austin,4,015,1513,60644
420403,<NA>,Garfield Ridge,56,Garfield Ridge,1,008,811,60638
5994681,<NA>,Austin,25,Austin,5,025,2532,60639
78639,<NA>,Wicker Park,24,West Town,5,014,1423,60622
169089,<NA>,West Ridge,2,West Ridge,3,024,2411,60645
442488,<NA>,Albany Park,14,Albany Park,5,017,1723,60625
262707,<NA>,Englewood,68,Englewood,1,007,712,60621
97642,<NA>,North Lawndale,29,North Lawndale,4,010,1022,60623
6007360,<NA>,West Pullman,53,West Pullman,2,005,523,60628
355360,<NA>,<NA>,<NA>,<NA>,3,024,2432,<NA>


In [79]:
# Finding Unique Key for Update
key_cols = ['neighborhood',	'community_code', 'community_name',	'sector', 'district', 'beat', 'zip_code']
target_col = 'ward'
# fill ward using Composite key
df_crime = geo.fill_from_composite_key(df_crime, key_cols, target_col)

--- Fill Summary: Column (ward) 530,713 missing values filled ---


In [80]:
# check
df_crime.loc[mask, cols].sample(n=10, random_state=SEED)

,ward,neighborhood,community_code,community_name,sector,district,beat,zip_code
6004402,29,Austin,25,Austin,4,015,1513,60644
420403,23,Garfield Ridge,56,Garfield Ridge,1,008,811,60638
5994681,37,Austin,25,Austin,5,025,2532,60639
78639,01,Wicker Park,24,West Town,5,014,1423,60622
169089,50,West Ridge,2,West Ridge,3,024,2411,60645
442488,33,Albany Park,14,Albany Park,5,017,1723,60625
262707,16,Englewood,68,Englewood,1,007,712,60621
97642,24,North Lawndale,29,North Lawndale,4,010,1022,60623
6007360,09,West Pullman,53,West Pullman,2,005,523,60628
355360,<NA>,<NA>,<NA>,<NA>,3,024,2432,<NA>


**Note:**
- Check East Garfield Park - Data inconsistencies
- Probamatic due to redistricting that happens every 10 years after the U.S. Census, or possible data

In [81]:
# examine community_name equal East Garfield Park
df_crime.loc[df_crime["community_name"].eq("East Garfield Park")] \
    .groupby(["ward", "community_code", "community_name", "zip_code"]) \
    .size()

ward  community_code  community_name      zip_code
02    27              East Garfield Park  60612       34779
                                          60624         311
      28              East Garfield Park  60612         102
06    44              East Garfield Park  60624           1
24    26              East Garfield Park  60624         444
      27              East Garfield Park  60612        6304
                                          60624       21767
      29              East Garfield Park  60612          29
                                          60624         986
27    23              East Garfield Park  60612          54
                                          60624          13
      26              East Garfield Park  60624         229
      27              East Garfield Park  60612       13493
                                          60624         841
      28              East Garfield Park  60612           7
28    23              East Garfield Park  60612  

In [82]:
# Only community_code is null & community_name not null
mask = df_crime.community_code.isna() & df_crime.community_name.notna()
print(f"NaN count: {(mask.sum()):,}\n")
df_crime.loc[mask, cols].head()

NaN count: 3



,ward,neighborhood,community_code,community_name,sector,district,beat,zip_code
146079,<NA>,O'Hare,<NA>,O'Hare,5,016,1614,60656
296133,<NA>,O'Hare,<NA>,O'Hare,5,016,1614,60656
5949801,<NA>,O'Hare,<NA>,O'Hare,5,016,1613,60656


In [83]:
# Fill NaNs for neighborhood & community_name using community_code
df_crime = geo.fill_geo_from_lookup(df_crime, 'community_name', ['community_code']).copy()

--- Fill Summary (Key: community_name) ---
Column 'community_code': 3 rows filled.


In [84]:
# Finding Unique Key for Update
key_cols = ['neighborhood',	'community_code', 'community_name',	'sector', 'district', 'beat', 'zip_code']
target_col = 'ward'
# fill ward using Composite key
df_crime = geo.fill_from_composite_key(df_crime, key_cols, target_col)

--- Fill Summary: Column (ward) 2 missing values filled ---


In [85]:
utils.any_nans(df_crime)

--- Missing Values Found (Total Rows: 8,469,872) ---
                       Count Percentage
neighborhood_area     116237    1.3724%
community_area        116237    1.3724%
zip_code_area         116181    1.3717%
x_coordinate           94303    1.1134%
y_coordinate           94303    1.1134%
ward                   84100    0.9929%
location_description   15338    0.1811%
community_code          9744    0.1150%
zip_code                9744    0.1150%
neighborhood            9744    0.1150%
community_name          9744    0.1150%
district                 139    0.0016%
district_location        139    0.0016%
sector                   139    0.0016%


## Total Time

In [86]:
# total elapsed time
elapsed = time.time() - start
print(f"{elapsed:.2f}s to process {df_crime.shape[0]:,} rows")

77.69s to process 8,469,872 rows


## Save using Native PyArrow IPC

In [87]:
# Convert Pandas DataFrame to PyArrow Table
table = pa.Table.from_pandas(df_crime)

# Save as Arrow IPC stream (faster than pickle)
with open('../Data/crime_data.arrow', 'wb') as f:
    with ipc.new_file(f, table.schema) as writer:
        writer.write_table(table)